# Larme 1B — QLoRA 训练（Kaggle T4 / Colab 免费）

基座：`Qwen/Qwen2.5-1.5B-Instruct`（真 1.5B 权重，非空壳）  
数据：`pairs.jsonl`（417 条指令对）  
产物：LoRA adapter + 合并后的 FP16 权重，下载回来在本机转 GGUF Q4 上架。

预计：T4 上 15–25 分钟。

In [ ]:
!pip install -q "transformers>=4.44" peft datasets bitsandbytes accelerate
import torch
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), "GiB")

In [ ]:
# 把仓库里的 qlora_7b.py 和 pairs.jsonl 放进工作目录（本 notebook 与它们同级）
!ls -lh pairs.jsonl qlora_7b.py

In [ ]:
!python qlora_7b.py \
  --base Qwen/Qwen2.5-1.5B-Instruct \
  --data pairs.jsonl \
  --out /kaggle/working/larme-1b \
  --epochs 3 \
  --lr 1e-4 \
  --batch 4 \
  --accum 4 \
  --max-len 768 \
  --merge

In [ ]:
!du -sh /kaggle/working/larme-1b /kaggle/working/larme-1b-merged 2>/dev/null
!cd /kaggle/working && zip -qr larme-1b.zip larme-1b larme-1b-merged && ls -lh larme-1b.zip

## 下载回来之后

1. 解压得到 `larme-1b-merged/`（FP16，约 3.1 GB）
2. 转 GGUF 并量化：
   ```bash
   python convert_hf_to_gguf.py larme-1b-merged --outfile larme-1b-f16.gguf
   ./llama-quantize larme-1b-f16.gguf larme-1b-q4_k_m.gguf Q4_K_M
   ```
3. 放到 `/home/agentuser/models/gguf/`，在 `models.json` 加一条，`-ultra/-max/-high` 三档自动可用。

> 说明：这是**微调**现成开源基座，不是从零预训 1B。报告里会写清基座归属与训练量。